# 5단계 — FastAPI 서버 & 사용 데모

학습된 모델을 REST API로 서빙합니다.

## 서버 실행 방법

터미널에서 프로젝트 루트(`IHateThis/`)로 이동 후:

```bash
uvicorn src.api.main:app --host 0.0.0.0 --port 8000 --reload
```

> ML 모델 먼저 학습(`python -m src.train --model ml`) 후 실행하세요.

## 1. 서버 상태 확인 (`GET /health`)

In [2]:
import requests
import json

BASE_URL = 'http://localhost:8000'

def pretty(response):
    print(f'HTTP {response.status_code}')
    print(json.dumps(response.json(), indent=2, ensure_ascii=False))

pretty(requests.get(f'{BASE_URL}/health'))

HTTP 200
{
  "status": "ok",
  "ml_model_loaded": true,
  "bert_model_loaded": true
}


## 2. 모델 정보 (`GET /model/info`)

In [3]:
pretty(requests.get(f'{BASE_URL}/model/info'))

HTTP 200
{
  "ml_model": "TF-IDF + Logistic Regression",
  "bert_model": "KR-ELECTRA (artifacts/bert_model)",
  "spam_threshold": 0.7
}


## 3. 단일 이메일 분류 (`POST /predict`)

In [4]:
spam_email = {
    'subject': '[긴급] 무료 대출 1억 — 지금 바로 신청하세요!',
    'body': '안녕하세요. 신용등급 상관없이 당일 대출 가능합니다. 지금 바로 클릭하세요! http://free-loan.co.kr',
    'model': 'ml'
}

pretty(requests.post(f'{BASE_URL}/predict', json=spam_email))

HTTP 200
{
  "label": "spam",
  "confidence": 0.8256850142097213,
  "spam_probability": 0.8256850142097213,
  "ham_probability": 0.1743149857902787,
  "model": "ml",
  "processing_time_ms": 1.2
}


In [5]:
ham_email = {
    'subject': '내일 오전 팀 회의 자료 공유',
    'body': '안녕하세요. 내일 오전 10시 팀 회의를 위한 자료를 공유드립니다. 첨부 파일 확인 부탁드립니다.',
    'model': 'ml'
}

pretty(requests.post(f'{BASE_URL}/predict', json=ham_email))

HTTP 200
{
  "label": "ham",
  "confidence": 0.5455426688157461,
  "spam_probability": 0.4544573311842539,
  "ham_probability": 0.5455426688157461,
  "model": "ml",
  "processing_time_ms": 1.39
}


## 4. 배치 분류 (`POST /predict/batch`)

In [6]:
batch_request = {
    'emails': [
        {'subject': '축하합니다! 1등 당첨!', 'body': '지금 바로 수령하세요. 010-0000-0000', 'model': 'ml'},
        {'subject': '월급명세서 안내', 'body': '이번 달 급여명세서를 첨부드립니다.', 'model': 'ml'},
        {'subject': '투자로 월 500만원 수익', 'body': '비밀 투자 전략 공개! 지금 가입하세요.', 'model': 'ml'},
        {'subject': '배송 완료 안내', 'body': '주문하신 상품이 배송 완료되었습니다.', 'model': 'ml'},
    ]
}

resp = requests.post(f'{BASE_URL}/predict/batch', json=batch_request)
data = resp.json()

print(f'총 {data["total"]}건 처리 | {data["total_processing_time_ms"]:.1f}ms\n')
for i, (email, result) in enumerate(zip(batch_request['emails'], data['results'])):
    label  = result['label'].upper()
    conf   = result['confidence']
    spam_p = result['spam_probability']
    bar = '█' * int(spam_p * 20)
    print(f'{i+1}. [{label:4s}] spam={spam_p:.2%} |{bar:<20}| {email["subject"]}')

총 4건 처리 | 1.0ms

1. [HAM ] spam=63.69% |████████████        | 축하합니다! 1등 당첨!
2. [HAM ] spam=44.99% |████████            | 월급명세서 안내
3. [HAM ] spam=67.93% |█████████████       | 투자로 월 500만원 수익
4. [HAM ] spam=69.07% |█████████████       | 배송 완료 안내


## 5. 스팸 임계값(threshold) 시각화

현재 임계값 0.7은 정상 메일이 스팸으로 잘못 분류되는 FP를 억제합니다.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False
from sklearn.metrics import precision_score, recall_score, f1_score

from src.data.preprocessor import Preprocessor
from src.data.loader import load_dataset
from src.models.ml_model import MLSpamClassifier

pre  = Preprocessor(use_konlpy=False)
data = load_dataset('../data/translated_en_to_ko.csv', pre)
model = MLSpamClassifier.load('../artifacts/ml_model.joblib')

probas = model.predict_proba(data.X_test)
spam_probs = np.array([p[1] for p in probas])
y_test = np.array(data.y_test)

thresholds = np.arange(0.3, 0.95, 0.05)
precisions, recalls, f1s = [], [], []

for th in thresholds:
    y_pred = (spam_probs >= th).astype(int)
    precisions.append(precision_score(y_test, y_pred, zero_division=0))
    recalls.append(recall_score(y_test, y_pred, zero_division=0))
    f1s.append(f1_score(y_test, y_pred, zero_division=0))

plt.figure(figsize=(9, 5))
plt.plot(thresholds, precisions, 'o-', label='Precision', color='tomato')
plt.plot(thresholds, recalls,   's-', label='Recall',    color='steelblue')
plt.plot(thresholds, f1s,       '^-', label='F1',        color='seagreen')
plt.axvline(x=0.7, color='gray', linestyle='--', label='현재 임계값 (0.7)')
plt.xlabel('스팸 임계값')
plt.ylabel('점수')
plt.title('임계값에 따른 성능 변화')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '../data/korean_spam_dataset.csv'

---
## 참고: 이 모델의 오버피팅 방지 적용 사항

| 항목 | 내용 |
|------|------|
| 학습 데이터 | `korean_spam_dataset.csv` + `claude_generated_ko.csv` (총 8,000개, 2개 출처) |
| ML 정규화 | Logistic Regression `class_weight='balanced'`, L2 (`C=1.0`) |
| BERT 정규화 | `weight_decay=0.01`, `dropout` (AutoModel 내장), gradient clipping |
| BERT Early Stopping | `early_stopping_patience=3` — val F1 개선 없으면 조기 종료 |
| 교차 검증 | 5-Fold Stratified CV (노트북 03 참조) |
| 전처리 | URL/이메일/전화번호 토큰화로 특정 값 암기 방지 |

자세한 내용은 프로젝트 루트의 `OVERFITTING.md`를 참조하세요.